# putEMG — Anatomical Validation of Channel Clusters

Cross-references the 8 representative channels selected by `clustering_features.ipynb`
against the known electrode-to-muscle-group mapping of the putEMG forearm array.

**Goal:** verify that the clustering algorithm recovers anatomically meaningful
channel groups, and produce a forearm diagram showing muscle regions, electrode
positions, and selected channels.

**Prerequisites:** run `clustering_features.ipynb` first to produce
`feat_representative_channels.npy` and `feat_cluster_labels.npy`.

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from PIL import Image

sys.path.append(os.path.abspath('../../../'))

In [2]:
CLUSTERING_DIR = os.path.abspath(os.getcwd())
RESULTS_DIR    = os.path.join(CLUSTERING_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

rep_path    = os.path.join(CLUSTERING_DIR, 'feat_representative_channels.npy')
labels_path = os.path.join(CLUSTERING_DIR, 'feat_cluster_labels.npy')

if not os.path.exists(rep_path):
    raise FileNotFoundError(
        'Missing: ' + rep_path + '. Run clustering_features.ipynb first.'
    )

REPRESENTATIVES = np.load(rep_path).tolist()
CLUSTER_LABELS  = np.load(labels_path).tolist()

N_CLUSTERS = len(REPRESENTATIVES)
N_CHANNELS = len(CLUSTER_LABELS)

print(f'Clusters : {N_CLUSTERS}')
print(f'Channels : {N_CHANNELS}')
print(f'Representatives (0-idx): {REPRESENTATIVES}')
print(f'Representatives (1-idx): {[c+1 for c in REPRESENTATIVES]}')

Clusters : 8
Channels : 24
Representatives (0-idx): [0, 3, 6, 14, 16, 18, 20, 22]
Representatives (1-idx): [1, 4, 7, 15, 17, 19, 21, 23]


In [ ]:
# Electrode layout for putEMG 24-channel array
# 3 elastic bands × 8 electrodes = 24 channels
# Angular spacing: 45° per electrode (360° / 8)
# Electrode 1 of each band placed over the ulna bone, numbered clockwise viewed distally
# Band assignment: elbow band = ch 1–8, middle = ch 9–16, wrist = ch 17–24
#
# Angle convention used here: 0° = Ulnar, increasing clockwise (matches paper ch-1 placement)
# Muscle compartment boundaries are shifted accordingly from standard dorsal-0 convention.
ANGLES_DEG = [i * 45 for i in range(8)] * 3   # 0°,45°,90°,...,315° repeated for all 3 rings
ROWS       = [0] * 8 + [1] * 8 + [2] * 8      # 0=elbow (proximal), 1=middle, 2=wrist (distal)

# Muscle compartment mapping (approximate sectors, 0° = Ulnar, clockwise viewed distally)
# Ulnar (0°) → Dorsal (90°) → Radial (180°) → Palmar (270°) → Ulnar (360°)
MUSCLES = [
    (315,  45, 'ECU / FCU',   'Extensor/Flexor Carpi Ulnaris',              '#F0B27A'),
    ( 45, 135, 'ECRL / BR',   'Extensor Carpi Radialis & Brachioradialis',  '#5DADE2'),
    (135, 225, 'FCR / Palm.', 'Flexor Carpi Radialis & Palmaris Longus',    '#AF7AC5'),
    (225, 315, 'FDS / FDP',   'Flexor Digitorum Superficialis & Profundus', '#EC7063'),
]

print('Electrode layout defined.')
print(f'  Ch 1–8   (elbow band,  proximal): angles {ANGLES_DEG[:8]} deg')
print(f'  Ch 9–16  (middle band, mid)     : angles {ANGLES_DEG[8:16]} deg')
print(f'  Ch 17–24 (wrist band,  distal)  : angles {ANGLES_DEG[16:]} deg')

In [ ]:
ASSETS_DIR = os.path.join(CLUSTERING_DIR, 'assets')
img_dorsal = np.array(Image.open(os.path.join(ASSETS_DIR, 'forearm_dorsal.png')))
img_palmar = np.array(Image.open(os.path.join(ASSETS_DIR, 'forearm_palmar.png')))

# ── Forearm anchor parameters (pixel coords in original full image) ──────────
#
# Angle convention: 0° = Ulnar, 90° = Dorsal, 180° = Radial, 270° = Palmar
# (matches paper: ch1 of each band placed over ulna = 0°, clockwise viewed distally)
#
# Projection formula: x = x_cen + sign * x_half * sin(radians(angle - a_ref))
#   where a_ref is the angle that lands at the horizontal center of each view.
#
# Gray418 (406×1241 px) — dorsal/extensor view, elbow at top, wrist at bottom
#   Dorsal (90°) → center; Ulnar (0°) → right; Radial (180°) → left
DORSAL = dict(
    x_cen   = 200,
    x_half  = 118,
    y_rings = [285, 400, 510],   # ring 0 (elbow), ring 1 (middle), ring 2 (wrist)
    y_crop  = (80, 760),
    a_ref   = 90,                # dorsal projects to center in this view
    sign    = +1,
)

# Gray414 (227×700 px) — palmar/flexor view, elbow at top, wrist at bottom
#   Palmar (270°) → center; Ulnar (0°) → left; Radial (180°) → right
PALMAR = dict(
    x_cen   = 113,
    x_half  = 82,
    y_rings = [220, 300, 385],
    y_crop  = (60, 570),
    a_ref   = 270,               # palmar projects to center in this view
    sign    = -1,
)

CLUSTER_COLORS = plt.cm.tab10(np.linspace(0, 1, N_CLUSTERS))

# ── Figure: proportional panel widths so neither image is stretched ──────────
d_h = DORSAL['y_crop'][1] - DORSAL['y_crop'][0]
p_h = PALMAR['y_crop'][1] - PALMAR['y_crop'][0]
FIG_H = 10.0
d_w = img_dorsal.shape[1] / d_h * FIG_H
p_w = img_palmar.shape[1] / p_h * FIG_H

fig, (ax_d, ax_p) = plt.subplots(
    1, 2, figsize=(d_w + p_w + 1.5, FIG_H),
    gridspec_kw={'width_ratios': [d_w, p_w]}
)
fig.patch.set_facecolor('white')

VIEWS = [
    (ax_d, 'Dorsal View — Extensors', img_dorsal, DORSAL,  90),
    (ax_p, 'Palmar View — Flexors',   img_palmar, PALMAR, 270),
]

for ax, title, img, params, front_angle in VIEWS:
    y0, y1 = params['y_crop']
    ax.imshow(img[y0:y1], aspect='equal')

    for ch_idx in range(N_CHANNELS):
        cl    = CLUSTER_LABELS[ch_idx]
        is_r  = ch_idx in REPRESENTATIVES
        angle = ANGLES_DEG[ch_idx]
        ring  = ROWS[ch_idx]
        a_rad = np.radians(angle - params['a_ref'])

        x_pix = params['x_cen'] + params['sign'] * params['x_half'] * np.sin(a_rad)
        y_pix = params['y_rings'][ring] - y0

        # Front face: channel within ±90° of this view's center angle
        angle_diff = (angle - front_angle + 180) % 360 - 180   # in [-180, 180]
        is_front   = abs(angle_diff) <= 90

        clr   = CLUSTER_COLORS[cl % N_CLUSTERS]
        alpha = 1.0 if is_front else 0.32

        ax.scatter(x_pix, y_pix,
                   c=[clr],
                   marker='*' if is_r else 'o',
                   s=420 if is_r else 120,
                   edgecolors='white' if is_front else '#444444',
                   linewidths=1.3,
                   zorder=20, alpha=alpha)
        ax.text(x_pix + 8, y_pix, str(ch_idx + 1),
                fontsize=6.5,
                color='#111111' if is_front else '#aaaaaa',
                fontweight='bold' if is_r else 'normal',
                va='center', zorder=21)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=7)
    ax.axis('off')

# ── Shared legend ─────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=CLUSTER_COLORS[k % N_CLUSTERS],
                   edgecolor='#333333', linewidth=0.5, label=f'Cluster {k}')
    for k in range(N_CLUSTERS)
]
legend_handles += [
    Line2D([0], [0], marker='*', color='w', markerfacecolor='gray',
           markeredgecolor='white', markersize=13, label='Representative channel'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markeredgecolor='white', markersize=8,  label='Non-representative'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#aaaaaa',
           markeredgecolor='#444444', markersize=8, alpha=0.5, label='Hidden (behind arm)'),
]
fig.legend(handles=legend_handles, loc='lower center',
           ncol=min(N_CLUSTERS + 3, 6), fontsize=9, frameon=True,
           bbox_to_anchor=(0.5, 0.0), borderpad=0.8)

fig.suptitle('putEMG Electrode Layout — Feature-Based Cluster Validation',
             fontsize=14, fontweight='bold', y=1.005)
fig.text(0.5, -0.015,
         "Anatomy: Gray's Anatomy 20th ed. (1918, public domain)  ·  ★ = cluster representative  ·  faded = behind arm",
         ha='center', fontsize=8, color='#666666', style='italic')

plt.tight_layout(rect=[0, 0.07, 1, 1.0])
plt.savefig(os.path.join(RESULTS_DIR, 'anatomical_layout.png'),
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved → results/anatomical_layout.png')

In [5]:
# Angular coverage of representative channels
print('Angular coverage of representative channels:')
print('-' * 55)
for k in range(N_CLUSTERS):
    members = [i for i, lbl in enumerate(CLUSTER_LABELS) if lbl == k]
    rep     = REPRESENTATIVES[k]
    angles  = [ANGLES_DEG[i] for i in members]
    print(f'  Cluster {k}: ch {[i+1 for i in members]}  '
          f'({angles} deg)  ->  rep: ch {rep+1} ({ANGLES_DEG[rep]} deg)')

rep_angles = sorted([ANGLES_DEG[r] for r in REPRESENTATIVES])
print()
print(f'Representative angles (sorted): {rep_angles}')
diffs = [rep_angles[i+1] - rep_angles[i] for i in range(len(rep_angles)-1)]
diffs.append(360 - rep_angles[-1] + rep_angles[0])
print(f'Inter-representative gaps (deg): {diffs}')
print(f'Max gap: {max(diffs)} deg  |  Min gap: {min(diffs)} deg')

Angular coverage of representative channels:
-------------------------------------------------------
  Cluster 0: ch [1, 9]  ([0, 240] deg)  ->  rep: ch 1 (0 deg)
  Cluster 1: ch [17]  ([120] deg)  ->  rep: ch 4 (90 deg)
  Cluster 2: ch [18, 19, 20]  ([150, 180, 210] deg)  ->  rep: ch 7 (180 deg)
  Cluster 3: ch [2, 3, 4, 5, 10, 11, 12]  ([30, 60, 90, 120, 270, 300, 330] deg)  ->  rep: ch 15 (60 deg)
  Cluster 4: ch [15, 16]  ([60, 90] deg)  ->  rep: ch 17 (120 deg)
  Cluster 5: ch [6, 7, 8]  ([150, 180, 210] deg)  ->  rep: ch 19 (180 deg)
  Cluster 6: ch [23, 24]  ([300, 330] deg)  ->  rep: ch 21 (240 deg)
  Cluster 7: ch [13, 14, 21, 22]  ([0, 30, 240, 270] deg)  ->  rep: ch 23 (300 deg)

Representative angles (sorted): [0, 60, 90, 120, 180, 180, 240, 300]
Inter-representative gaps (deg): [60, 30, 30, 60, 0, 60, 60, 60]
Max gap: 60 deg  |  Min gap: 0 deg


In [6]:
# Which anatomical region does each cluster span?
def get_muscle(angle_deg):
    for start, end, short, long, color in MUSCLES:
        if start <= angle_deg < end:
            return short
    return 'Unknown'

print('Cluster composition by anatomical region:')
print('=' * 65)
for k in range(N_CLUSTERS):
    members = [i for i, lbl in enumerate(CLUSTER_LABELS) if lbl == k]
    rep     = REPRESENTATIVES[k]
    regions = list(dict.fromkeys(get_muscle(ANGLES_DEG[i]) for i in members))
    rows_in = sorted(set(ROWS[i] for i in members))
    row_str = ' + '.join(['proximal' if r == 0 else 'distal' for r in rows_in])
    print(f'Cluster {k}  (rep: ch {rep+1:2d}, {ANGLES_DEG[rep]:3d} deg)  '
          f'members: {[i+1 for i in members]}')
    print(f'         regions: {regions}  |  rows: {row_str}')
    print()

Cluster composition by anatomical region:
Cluster 0  (rep: ch  1,   0 deg)  members: [1, 9]
         regions: ['ECRL / BR', 'FDS / FDP']  |  rows: proximal

Cluster 1  (rep: ch  4,  90 deg)  members: [17]
         regions: ['Ext. Dig.']  |  rows: distal

Cluster 2  (rep: ch  7, 180 deg)  members: [18, 19, 20]
         regions: ['ECU / FCU', 'FDS / FDP']  |  rows: distal

Cluster 3  (rep: ch 15,  60 deg)  members: [2, 3, 4, 5, 10, 11, 12]
         regions: ['ECRL / BR', 'Ext. Dig.', 'FDS / FDP', 'FCR / Palm.']  |  rows: proximal

Cluster 4  (rep: ch 17, 120 deg)  members: [15, 16]
         regions: ['Ext. Dig.']  |  rows: distal

Cluster 5  (rep: ch 19, 180 deg)  members: [6, 7, 8]
         regions: ['ECU / FCU', 'FDS / FDP']  |  rows: proximal

Cluster 6  (rep: ch 21, 240 deg)  members: [23, 24]
         regions: ['FCR / Palm.']  |  rows: distal

Cluster 7  (rep: ch 23, 300 deg)  members: [13, 14, 21, 22]
         regions: ['ECRL / BR', 'FDS / FDP']  |  rows: distal

